In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Any
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
class WeatherFeatureEngineer:
    def __init__(self):
        self.cols_to_drop = [
            'dew_point_2m_max (°C)',
            'wind_speed_10m_max (km/h)',
            'surface_pressure_max (hPa)',
            'surface_pressure_min (hPa)',
            'wet_bulb_temperature_2m_max (°C)',
            'wet_bulb_temperature_2m_min (°C)',
            'soil_temperature_0_to_100cm_mean (°C)'
        ]
        self.low_importance_features = [
            'et0_fao_evapotranspiration_sum (mm)'
        ]
    
    def create_temporal_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df['day'] = df['date'].dt.day
        df['month'] = df['date'].dt.month
        df['year'] = df['date'].dt.year
        df['dayofweek'] = df['date'].dt.dayofweek
        df['is_weekend'] = df['dayofweek'] >= 5
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['dayofyear'] = df['date'].dt.dayofyear
        df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
        df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
        return df

    def create_additional_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df['temp_range'] = df['temperature_2m_max (°C)'] - df['temperature_2m_min (°C)']
        df['wind_gust_range'] = df['wind_gusts_10m_max (km/h)'] - df['wind_gusts_10m_min (km/h)']
        df['avg_wind_speed'] = (df['wind_speed_10m_max (km/h)'] + df['wind_speed_10m_min (km/h)']) / 2
        df['wind_variability'] = df['wind_speed_10m_max (km/h)'] - df['wind_speed_10m_min (km/h)']
        df['humidity_range'] = df['relative_humidity_2m_max (%)'] - df['relative_humidity_2m_min (%)']
        df['dew_point_range'] = df['dew_point_2m_max (°C)'] - df['dew_point_2m_min (°C)']
        df['sunshine_ratio'] = df['sunshine_duration (s)'] / df['daylight_duration (s)']
        df['rain_today'] = df['rain_sum (mm)'].apply(lambda x: 1 if x > 0 else 0)
        df['pressure_range'] = df['surface_pressure_max (hPa)'] - df['surface_pressure_min (hPa)']
        df['daylight_to_sunshine_ratio'] = df['daylight_duration (s)'] / (df['sunshine_duration (s)'] + 1)
        df.ffill(inplace=True)
        return df

    def create_targeted_weather_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)

        # Tier 1
        tier1 = ['cloud_cover_max (%)', 'rain_sum (mm)', 'precipitation_hours (h)', 'cloud_cover_min (%)']
        for col in tier1:
            if col in df.columns:
                c = col.replace(' ', '_').replace('(%)', '').replace('(mm)', '').replace('(h)', '')
                for lag in [1, 2, 3, 7]:
                    df[f'{c}_lag_{lag}'] = df[col].shift(lag)
                for win in [3, 7, 14]:
                    df[f'{c}_rolling_mean_{win}d'] = df[col].rolling(win, min_periods=1).mean()
                    df[f'{c}_rolling_std_{win}d'] = df[col].rolling(win, min_periods=1).std()
                    df[f'{c}_rolling_max_{win}d'] = df[col].rolling(win, min_periods=1).max()
                df[f'{c}_3day_trend'] = df[col].diff(3)
                df[f'{c}_7day_trend'] = df[col].diff(7)
                df[f'{c}_ewm_3d'] = df[col].ewm(span=3, min_periods=1).mean()
                df[f'{c}_ewm_7d'] = df[col].ewm(span=7, min_periods=1).mean()

        # Rain Today (binary)
        if 'rain_today' in df.columns:
            df['rain_today_lag_1'] = df['rain_today'].shift(1)
            df['rain_today_lag_2'] = df['rain_today'].shift(2)
            df['rain_today_lag_3'] = df['rain_today'].shift(3)
            df['rain_frequency_7d'] = df['rain_today'].rolling(7, min_periods=1).mean()
            df['rain_frequency_14d'] = df['rain_today'].rolling(14, min_periods=1).mean()
            df['consecutive_rain_days'] = (
                df['rain_today'].groupby((df['rain_today'] != df['rain_today'].shift()).cumsum()).cumsum()
            )
            rain_idx = df[df['rain_today'] == 1].index
            df['days_since_rain'] = np.nan
            for i in df.index:
                prev = rain_idx[rain_idx < i]
                if len(prev) > 0:
                    df.loc[i, 'days_since_rain'] = i - prev.max()
            df['days_since_rain'] = df['days_since_rain'].fillna(30)

        # Tier 2
        tier2 = ['temp_range', 'relative_humidity_2m_min (%)', 'temperature_2m_max (°C)', 'humidity_range', 'relative_humidity_2m_max (%)']
        for col in tier2:
            if col in df.columns:
                c = col.replace(' ', '_').replace('(%)', '').replace('(°C)', '')
                for lag in [1, 3, 7]:
                    df[f'{c}_lag_{lag}'] = df[col].shift(lag)
                df[f'{c}_rolling_mean_7d'] = df[col].rolling(7, min_periods=1).mean()
                df[f'{c}_rolling_std_7d'] = df[col].rolling(7, min_periods=1).std()
                df[f'{c}_7day_trend'] = df[col].diff(7)

        # Interactions
        if 'cloud_cover_max (%)' in df.columns and 'rain_sum (mm)' in df.columns:
            df['cloud_rain_interaction'] = df['cloud_cover_max (%)'] * df['rain_sum (mm)']
        if 'temp_range' in df.columns and 'relative_humidity_2m_min (%)' in df.columns:
            df['temp_humidity_interaction'] = df['temp_range'] * df['relative_humidity_2m_min (%)']
        if 'precipitation_hours (h)' in df.columns and 'cloud_cover_max (%)' in df.columns:
            df['precip_cloud_interaction'] = df['precipitation_hours (h)'] * df['cloud_cover_max (%)']

        # Weather pattern indicators
        if 'cloud_cover_max (%)' in df.columns:
            df['cloud_volatility_7d'] = df['cloud_cover_max (%)'].rolling(7, min_periods=1).std()
        if 'rain_sum (mm)' in df.columns:
            df['rain_volatility_7d'] = df['rain_sum (mm)'].rolling(7, min_periods=1).std()
        if 'temperature_2m_max (°C)' in df.columns and 'dayofyear' in df.columns:
            df['temp_seasonal_anomaly'] = (
                df['temperature_2m_max (°C)'] - df.groupby('dayofyear')['temperature_2m_max (°C)'].transform('mean')
            )

        # Drop low-importance features
        time_related = ['year', 'month', 'day', 'dayof', 'weekend', 'sin', 'cos']
        drop_final = [
            f for f in self.low_importance_features
            if f in df.columns and not any(k in f.lower() for k in time_related)
        ]
        df.drop(columns=drop_final, inplace=True, errors='ignore')

        return df

    def drop_unimportant_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop(columns=self.cols_to_drop, errors='ignore')

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = self.create_temporal_features(df)
        df = self.create_additional_features(df)
        df = self.drop_unimportant_columns(df)
        df = self.create_targeted_weather_features(df)
        return df


    def generate_model_inputs(self, df: pd.DataFrame, model_type: str, scaler_X=None):
        df = df.copy()

        if model_type == 'weather_condition':
            # --- Map weather_code to descriptive weather_condition strings ---
            weather_conditions_map = {
                0: "Clear Sky ☀️",
                1: "Mainly Clear 🌤",
                2: "Partly Cloudy ⛅",
                3: "Cloudy ☁️",
                45: "Fog 🌫",
                48: "Fog 🌫",
                51: "Light Drizzle 🌦",
                53: "Moderate Drizzle 🌧",
                55: "Heavy Drizzle 🌧",
                61: "Light Rain 🌦",
                63: "Moderate Rain 🌧",
                65: "Heavy Rain 🌧",
                71: "Light Snow ❄️",
                73: "Moderate Snow ❄️",
                75: "Heavy Snow ❄️",
                95: "Thunderstorm ⛈"
            }

            if 'weather_code' in df.columns:
                df['weather_condition'] = df['weather_code'].map(weather_conditions_map)
                df = df.drop(columns=['weather_code'])
            
            # -------------------------------
            # 🌤 SPECIAL CASE: Classification
            # -------------------------------
            df['weather_condition'] = df['weather_condition'].replace({
                'Mainly Clear 🌤': 'Partly Clear 🌤/⛅',
                'Partly Cloudy ⛅': 'Partly Clear 🌤/⛅'
            })

            df = df.dropna(subset=['weather_condition'])

            # Fill missing numerics
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            for col in numeric_cols:
                df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
                df[col] = df[col].interpolate().fillna(df[col].median())

            # Encode labels
            label_encoder = LabelEncoder()
            encoded_col = 'weather_condition_encoded'
            df[encoded_col] = label_encoder.fit_transform(df['weather_condition'])

            # Compute class weights
            unique_classes = df[encoded_col].unique()
            class_weights = compute_class_weight('balanced', classes=unique_classes, y=df[encoded_col])
            class_weight_dict = dict(zip(unique_classes, class_weights))

            # Select features (exclude date + original & encoded target)
            feature_cols = [col for col in df.columns if col not in ['date', 'weather_condition', encoded_col]]

            # Scale features
            scaler_X = StandardScaler()
            df[feature_cols] = scaler_X.fit_transform(df[feature_cols])

            X = df[feature_cols].values
            y = df[encoded_col].values

            return X, y, feature_cols, label_encoder, class_weight_dict, scaler_X

        else:
            # Regression models remain unchanged
            model_exclude_map = {
                'temp_max': ['date', 'weather_condition', 'relative_humidity_2m_max (%)',
                            'relative_humidity_2m_min (%)', 'temperature_2m_min (°C)', 'temperature_2m_max (°C)'],
                
                'temp_min': ['date', 'weather_condition', 'temperature_2m_max (°C)', 
                            'relative_humidity_2m_max (%)', 'relative_humidity_2m_min (%)', 'temperature_2m_min (°C)'],
                
                'humidity_max': ['date', 'weather_condition', 'temperature_2m_max (°C)', 'temperature_2m_min (°C)',
                                'relative_humidity_2m_min (%)', 'relative_humidity_2m_max (%)'],
                
                'humidity_min': ['date', 'weather_condition', 'temperature_2m_max (°C)', 'temperature_2m_min (°C)',
                                'relative_humidity_2m_min (%)', 'relative_humidity_2m_max (%)']
            }

            if model_type not in model_exclude_map:
                raise ValueError(f"Unknown model_type: {model_type}")

            exclude_cols = model_exclude_map[model_type]
            feature_cols = [col for col in df.columns if col not in exclude_cols]

            # Clean missing values
            df[feature_cols] = df[feature_cols].fillna(method='ffill').fillna(method='bfill').interpolate().fillna(df[feature_cols].median())

            # Scale if scaler provided
            if scaler_X:
                X_scaled = scaler_X.transform(df[feature_cols])
                return X_scaled, feature_cols
            else:
                return df[feature_cols].values, feature_cols


In [7]:
if __name__ == "__main__":
    import pandas as pd
    
    # Load raw data
    df = pd.read_csv("test_data.csv")
    
    # Initialize feature engineer and transform raw data
    engineer = WeatherFeatureEngineer()
    df_transformed = engineer.transform(df)
    
    # Define your models to prepare inputs for
    models = ['temp_max', 'temp_min', 'humidity_max', 'humidity_min', 'weather_condition']
    
    # Storage for processed inputs
    processed_data = {}
    
    for model_type in models:
        print(f"\nProcessing data for model: {model_type}")
        
        # For weather_condition (classification), get full tuple
        if model_type == 'weather_condition':
            X, y, feature_cols, label_encoder, class_weight_dict, scaler = engineer.generate_model_inputs(df_transformed, model_type)
            processed_data[model_type] = {
                'X': X,
                'y': y,
                'feature_cols': feature_cols,
                'label_encoder': label_encoder,
                'class_weight_dict': class_weight_dict,
                'scaler': scaler,
            }
            print(f" - Classes: {label_encoder.classes_}")
            print(f" - Features ({len(feature_cols)}): {feature_cols[:5]} ...")
        
        # For regression models, just get scaled X and feature columns
        else:
            X_scaled, feature_cols = engineer.generate_model_inputs(df_transformed, model_type)
            processed_data[model_type] = {
                'X': X_scaled,
                'feature_cols': feature_cols,
            }
            print(f" - Features ({len(feature_cols)}): {feature_cols[:5]} ...")
    
    # Optional: save transformed full dataframe for reference
    df_transformed.to_csv("processed_weather_data.csv", index=False)
    
    print("\n✅ All model inputs prepared and ready for training or inference.")



Processing data for model: temp_max
 - Features (145): ['weather_code (wmo code)', 'daylight_duration (s)', 'sunshine_duration (s)', 'rain_sum (mm)', 'precipitation_hours (h)'] ...

Processing data for model: temp_min
 - Features (145): ['weather_code (wmo code)', 'daylight_duration (s)', 'sunshine_duration (s)', 'rain_sum (mm)', 'precipitation_hours (h)'] ...

Processing data for model: humidity_max
 - Features (145): ['weather_code (wmo code)', 'daylight_duration (s)', 'sunshine_duration (s)', 'rain_sum (mm)', 'precipitation_hours (h)'] ...

Processing data for model: humidity_min
 - Features (145): ['weather_code (wmo code)', 'daylight_duration (s)', 'sunshine_duration (s)', 'rain_sum (mm)', 'precipitation_hours (h)'] ...

Processing data for model: weather_condition


KeyError: 'weather_condition'

In [ ]:
class AgriculturalRecommendationSystem:
    def __init__(self):
        """
        Initialize the Agricultural Recommendation System
        """
        self.weather_conditions = {
            'Clear Sky': '☀️',
            'Cloudy': '☁️', 
            'Heavy Drizzle': '🌧',
            'Heavy Rain': '🌧',
            'Light Drizzle': '🌦',
            'Light Rain': '🌦',
            'Moderate Drizzle': '🌧',
            'Moderate Rain': '🌧',
            'Partly Clear': '🌤'
        }
        
        # Agricultural knowledge base
        self.crop_requirements = {
            'tomatoes': {'temp_min': 15, 'temp_max': 30, 'humidity_min': 40, 'humidity_max': 70},
            'lettuce': {'temp_min': 10, 'temp_max': 25, 'humidity_min': 50, 'humidity_max': 80},
            'corn': {'temp_min': 18, 'temp_max': 35, 'humidity_min': 45, 'humidity_max': 75},
            'wheat': {'temp_min': 12, 'temp_max': 28, 'humidity_min': 40, 'humidity_max': 65},
            'rice': {'temp_min': 20, 'temp_max': 35, 'humidity_min': 70, 'humidity_max': 90},
            'beans': {'temp_min': 16, 'temp_max': 30, 'humidity_min': 50, 'humidity_max': 75},
            'carrots': {'temp_min': 8, 'temp_max': 24, 'humidity_min': 45, 'humidity_max': 70},
            'potatoes': {'temp_min': 12, 'temp_max': 26, 'humidity_min': 60, 'humidity_max': 80}
        }
        
    def load_models(self, model_paths: Dict[str, str]):
        """
        Load your trained LSTM models
        
        Args:
            model_paths: Dictionary with keys ['weather_condition', 'temp_max', 'temp_min', 'humidity_max', 'humidity_min']
                        and values as file paths to your saved models
        """
        self.models = {}
        # In your actual implementation, load your Keras/TensorFlow models
        # Example:
        # from tensorflow.keras.models import load_model
        # for model_name, path in model_paths.items():
        #     self.models[model_name] = load_model(path)
        
        # For demonstration, we'll simulate model loading
        for model_name in model_paths.keys():
            self.models[model_name] = f"Loaded model: {model_name}"
        
        print("Models loaded successfully!")
    
    def predict_weather(self, input_data: np.ndarray, days_ahead: int = 7) -> Dict[str, Any]:
        """
        Generate weather predictions using your LSTM models
        
        Args:
            input_data: Historical weather data for model input
            days_ahead: Number of days to predict
            
        Returns:
            Dictionary containing predictions for each weather parameter
        """
        predictions = {}
        
        # In your actual implementation, use your loaded models to predict
        # Example:
        # predictions['weather_condition'] = self.models['weather_condition'].predict(input_data)
        # predictions['temp_max'] = self.models['temp_max'].predict(input_data)
        # etc.
        
        # For demonstration, we'll simulate predictions
        weather_conditions = list(self.weather_conditions.keys())
        
        predictions = {
            'weather_condition': np.random.choice(weather_conditions, days_ahead),
            'temp_max': np.random.uniform(20, 35, days_ahead),
            'temp_min': np.random.uniform(10, 25, days_ahead),
            'humidity_max': np.random.uniform(60, 90, days_ahead),
            'humidity_min': np.random.uniform(30, 60, days_ahead)
        }
        
        return predictions
    
    def analyze_irrigation_needs(self, weather_condition: str, humidity_min: float, 
                               humidity_max: float, temp_max: float) -> Dict[str, Any]:
        """
        Analyze irrigation requirements based on weather predictions
        """
        irrigation_advice = {
            'priority': 'medium',
            'frequency': 'normal',
            'amount': 'moderate',
            'timing': 'morning',
            'notes': []
        }
        
        # Analyze based on weather condition
        if weather_condition in ['Heavy Rain', 'Moderate Rain', 'Heavy Drizzle', 'Moderate Drizzle']:
            irrigation_advice['priority'] = 'low'
            irrigation_advice['frequency'] = 'reduced'
            irrigation_advice['amount'] = 'minimal'
            irrigation_advice['notes'].append('Natural rainfall expected - reduce irrigation')
            
        elif weather_condition in ['Clear Sky', 'Partly Clear']:
            if temp_max > 30:
                irrigation_advice['priority'] = 'high'
                irrigation_advice['frequency'] = 'increased'
                irrigation_advice['amount'] = 'generous'
                irrigation_advice['notes'].append('Hot and sunny - increase watering')
            
        elif weather_condition == 'Cloudy':
            irrigation_advice['priority'] = 'medium'
            irrigation_advice['notes'].append('Moderate conditions - maintain regular schedule')
        
        # Adjust based on humidity
        if humidity_min < 40:
            irrigation_advice['priority'] = 'high'
            irrigation_advice['notes'].append('Low humidity - plants need extra water')
        elif humidity_max > 80:
            irrigation_advice['frequency'] = 'reduced'
            irrigation_advice['notes'].append('High humidity - reduce watering frequency')
        
        return irrigation_advice
    
    def analyze_pest_disease_risk(self, weather_condition: str, humidity_max: float, 
                                temp_max: float, temp_min: float) -> Dict[str, Any]:
        """
        Assess pest and disease risk based on weather conditions
        """
        risk_assessment = {
            'overall_risk': 'medium',
            'pest_risk': 'medium',
            'disease_risk': 'medium',
            'specific_threats': [],
            'preventive_measures': []
        }
        
        # High humidity and warm temperatures increase disease risk
        if humidity_max > 75 and temp_max > 25:
            risk_assessment['disease_risk'] = 'high'
            risk_assessment['overall_risk'] = 'high'
            risk_assessment['specific_threats'].extend(['Fungal diseases', 'Bacterial infections'])
            risk_assessment['preventive_measures'].extend([
                'Improve air circulation',
                'Apply preventive fungicides',
                'Monitor for early symptoms'
            ])
        
        # Rain conditions can increase disease pressure
        if weather_condition in ['Heavy Rain', 'Moderate Rain', 'Heavy Drizzle']:
            risk_assessment['disease_risk'] = 'high'
            risk_assessment['specific_threats'].append('Soil-borne diseases')
            risk_assessment['preventive_measures'].append('Ensure proper drainage')
        
        # Hot, dry conditions may increase pest activity
        if weather_condition == 'Clear Sky' and temp_max > 30 and humidity_max < 50:
            risk_assessment['pest_risk'] = 'high'
            risk_assessment['specific_threats'].extend(['Aphids', 'Spider mites', 'Thrips'])
            risk_assessment['preventive_measures'].extend([
                'Monitor for pest presence',
                'Maintain adequate moisture',
                'Consider beneficial insects'
            ])
        
        return risk_assessment
    
    def recommend_planting_activities(self, weather_condition: str, temp_max: float, 
                                    temp_min: float, humidity_avg: float) -> Dict[str, Any]:
        """
        Recommend planting and field activities based on weather
        """
        recommendations = {
            'planting_suitability': 'moderate',
            'recommended_crops': [],
            'field_activities': [],
            'avoid_activities': [],
            'timing_advice': []
        }
        
        # Analyze temperature conditions
        if temp_min < 10:
            recommendations['planting_suitability'] = 'poor'
            recommendations['avoid_activities'].append('Planting heat-sensitive crops')
            recommendations['timing_advice'].append('Wait for warmer conditions')
        elif temp_min > 15 and temp_max < 30:
            recommendations['planting_suitability'] = 'excellent'
            recommendations['recommended_crops'].extend(['tomatoes', 'beans', 'corn'])
        
        # Weather-specific recommendations
        if weather_condition in ['Heavy Rain', 'Moderate Rain']:
            recommendations['avoid_activities'].extend([
                'Planting seeds (risk of rot)',
                'Harvesting',
                'Soil cultivation'
            ])
            recommendations['timing_advice'].append('Wait for soil to dry before field work')
            
        elif weather_condition == 'Clear Sky':
            recommendations['field_activities'].extend([
                'Planting',
                'Transplanting',
                'Harvesting',
                'Soil preparation'
            ])
            recommendations['timing_advice'].append('Ideal conditions for most field activities')
        
        # Recommend suitable crops based on conditions
        for crop, requirements in self.crop_requirements.items():
            if (requirements['temp_min'] <= temp_min <= requirements['temp_max'] and
                requirements['humidity_min'] <= humidity_avg <= requirements['humidity_max']):
                if crop not in recommendations['recommended_crops']:
                    recommendations['recommended_crops'].append(crop)
        
        return recommendations
    
    def generate_comprehensive_report(self, predictions: Dict[str, Any], 
                                    days_ahead: int = 7) -> str:
        """
        Generate a comprehensive agricultural recommendation report
        """
        report = []
        report.append("🌱 AGRICULTURAL WEATHER RECOMMENDATION REPORT 🌱")
        report.append("=" * 50)
        report.append(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report.append(f"Forecast period: {days_ahead} days\n")
        
        # Daily analysis
        for day in range(days_ahead):
            day_date = datetime.now() + timedelta(days=day+1)
            weather_cond = predictions['weather_condition'][day]
            temp_max = predictions['temp_max'][day]
            temp_min = predictions['temp_min'][day]
            humidity_max = predictions['humidity_max'][day]
            humidity_min = predictions['humidity_min'][day]
            humidity_avg = (humidity_max + humidity_min) / 2
            
            report.append(f"📅 DAY {day+1} - {day_date.strftime('%A, %B %d')}")
            report.append("-" * 30)
            report.append(f"Weather: {weather_cond} {self.weather_conditions[weather_cond]}")
            report.append(f"Temperature: {temp_min:.1f}°C - {temp_max:.1f}°C")
            report.append(f"Humidity: {humidity_min:.1f}% - {humidity_max:.1f}%\n")
            
            # Irrigation analysis
            irrigation = self.analyze_irrigation_needs(weather_cond, humidity_min, humidity_max, temp_max)
            report.append(f"💧 IRRIGATION ADVICE:")
            report.append(f"  Priority: {irrigation['priority'].upper()}")
            report.append(f"  Frequency: {irrigation['frequency']}")
            report.append(f"  Amount: {irrigation['amount']}")
            report.append(f"  Best timing: {irrigation['timing']}")
            for note in irrigation['notes']:
                report.append(f"  • {note}")
            report.append("")
            
            # Pest and disease risk
            risk = self.analyze_pest_disease_risk(weather_cond, humidity_max, temp_max, temp_min)
            report.append(f"🐛 PEST & DISEASE RISK:")
            report.append(f"  Overall risk: {risk['overall_risk'].upper()}")
            report.append(f"  Pest risk: {risk['pest_risk']}")
            report.append(f"  Disease risk: {risk['disease_risk']}")
            if risk['specific_threats']:
                report.append(f"  Threats: {', '.join(risk['specific_threats'])}")
            if risk['preventive_measures']:
                report.append(f"  Prevention:")
                for measure in risk['preventive_measures']:
                    report.append(f"    • {measure}")
            report.append("")
            
            # Planting recommendations
            planting = self.recommend_planting_activities(weather_cond, temp_max, temp_min, humidity_avg)
            report.append(f"🌱 PLANTING & FIELD ACTIVITIES:")
            report.append(f"  Planting conditions: {planting['planting_suitability'].upper()}")
            if planting['recommended_crops']:
                report.append(f"  Suitable crops: {', '.join(planting['recommended_crops'])}")
            if planting['field_activities']:
                report.append(f"  Recommended activities: {', '.join(planting['field_activities'])}")
            if planting['avoid_activities']:
                report.append(f"  Avoid: {', '.join(planting['avoid_activities'])}")
            for advice in planting['timing_advice']:
                report.append(f"  • {advice}")
            
            report.append("\n" + "="*50 + "\n")
        
        # Weekly summary
        report.append("📊 WEEKLY SUMMARY & RECOMMENDATIONS")
        report.append("=" * 50)
        
        # Calculate weekly averages
        avg_temp_max = np.mean(predictions['temp_max'])
        avg_temp_min = np.mean(predictions['temp_min'])
        avg_humidity = np.mean([predictions['humidity_max'], predictions['humidity_min']])
        
        rainy_days = sum(1 for cond in predictions['weather_condition'] 
                        if 'Rain' in cond or 'Drizzle' in cond)
        
        report.append(f"Average temperatures: {avg_temp_min:.1f}°C - {avg_temp_max:.1f}°C")
        report.append(f"Average humidity: {avg_humidity:.1f}%")
        report.append(f"Rainy days expected: {rainy_days}/{days_ahead}")
        report.append("")
        
        # Weekly recommendations
        report.append("🎯 KEY WEEKLY RECOMMENDATIONS:")
        if rainy_days >= 3:
            report.append("• High rainfall expected - focus on drainage and disease prevention")
        if avg_temp_max > 30:
            report.append("• Hot week ahead - increase irrigation and provide shade where possible")
        if avg_humidity > 75:
            report.append("• High humidity - monitor closely for fungal diseases")
        if rainy_days <= 1 and avg_temp_max > 25:
            report.append("• Dry conditions - excellent for harvesting and field work")
        
        report.append("\n" + "="*50)
        report.append("Generated by Agricultural Weather Recommendation System")
        
        return "\n".join(report)
    
    def get_recommendations(self, input_data: np.ndarray, days_ahead: int = 7) -> str:
        """
        Main method to get comprehensive agricultural recommendations
        
        Args:
            input_data: Historical weather data for LSTM model input
            days_ahead: Number of days to predict and analyze
            
        Returns:
            Comprehensive recommendation report as string
        """
        # Get predictions from your models
        predictions = self.predict_weather(input_data, days_ahead)
        
        # Generate comprehensive report
        report = self.generate_comprehensive_report(predictions, days_ahead)
        
        return report

# Example usage
if __name__ == "__main__":
    # Initialize the recommendation system
    agri_system = AgriculturalRecommendationSystem()
    
    # Load your trained models (replace with actual paths)
    model_paths = {
        'weather_condition': 'path/to/weather_condition_model.h5',
        'temp_max': 'path/to/temp_max_model.h5',
        'temp_min': 'path/to/temp_min_model.h5',
        'humidity_max': 'path/to/humidity_max_model.h5',
        'humidity_min': 'path/to/humidity_min_model.h5'
    }
    
    # Load models (uncomment when you have actual model files)
    # agri_system.load_models(model_paths)
    
    # Simulate input data (replace with your actual historical data)
    # This should match the input shape your LSTM models expect
    input_data = np.random.random((1, 30, 5))  # Example: 1 sample, 30 timesteps, 5 features
    
    # Get recommendations
    recommendations = agri_system.get_recommendations(input_data, days_ahead=7)
    
    # Print the report
    print(recommendations)
    
    # You can also save the report to a file
    with open('agricultural_recommendations.txt', 'w') as f:
        f.write(recommendations)
    
    print("\n📄 Report saved to 'agricultural_recommendations.txt'")